In [0]:
from pyspark.sql import functions as F

inventory_silver = spark.table("workspace.fashion_silver.inventory")
products_silver = spark.table("workspace.fashion_silver.products")

print("Inventory records:", inventory_silver.count())
print("Product records:", products_silver.count())

Inventory records: 40000
Product records: 10000


In [0]:
inventory_silver.groupBy("inventory_status").agg(
    F.count("*").alias("total_products"),
    F.sum("stock_quantity").alias("total_stock_units"),
    F.round(F.sum("inventory_value"), 2).alias("inventory_value")
).orderBy(F.desc("total_products")).show(truncate=False)

+----------------+--------------+-----------------+-----------------+
|inventory_status|total_products|total_stock_units|inventory_value  |
+----------------+--------------+-----------------+-----------------+
|In Stock        |35054         |9790121          |3.057365775131E10|
|Low Stock       |4868          |172001           |5.3637939174E8   |
|Out Of Stock    |78            |0                |0.0              |
+----------------+--------------+-----------------+-----------------+



In [0]:
warehouse_summary = (
    inventory_silver
    .groupBy("warehouse_id")
    .agg(
        F.count("*").alias("total_products"),
        F.sum("stock_quantity").alias("total_stock_units"),
        F.sum(
            F.when(F.col("inventory_status") == "In Stock", 1).otherwise(0)
        ).alias("in_stock_products"),
        F.sum(
            F.when(F.col("inventory_status") == "Low Stock", 1).otherwise(0)
        ).alias("low_stock_products"),
        F.sum(
            F.when(F.col("inventory_status") == "Out Of Stock", 1).otherwise(0)
        ).alias("out_of_stock_products"),
        F.round(F.sum("inventory_value"), 2).alias("inventory_value")
    )
    .orderBy("warehouse_id")
)

warehouse_summary.show(truncate=False)

+------------+--------------+-----------------+-----------------+------------------+---------------------+---------------+
|warehouse_id|total_products|total_stock_units|in_stock_products|low_stock_products|out_of_stock_products|inventory_value|
+------------+--------------+-----------------+-----------------+------------------+---------------------+---------------+
|WH001       |10000         |2478646          |8699             |1282              |19                   |7.79406648579E9|
|WH002       |10000         |2489808          |8764             |1210              |26                   |7.72768105369E9|
|WH003       |10000         |2488973          |8783             |1197              |20                   |7.76276509471E9|
|WH004       |10000         |2504695          |8808             |1179              |13                   |7.82552450886E9|
+------------+--------------+-----------------+-----------------+------------------+---------------------+---------------+



In [0]:
inventory_category = (
    inventory_silver
    .join(
        products_silver.select("product_id", "category"),
        on="product_id",
        how="inner"
    )
    .groupBy("category")
    .agg(
        F.countDistinct("product_id").alias("total_products"),
        F.sum("stock_quantity").alias("total_stock_units"),
        F.sum(
            F.when(F.col("inventory_status") == "Low Stock", 1).otherwise(0)
        ).alias("low_stock_records"),
        F.sum(
            F.when(F.col("inventory_status") == "Out Of Stock", 1).otherwise(0)
        ).alias("out_of_stock_records"),
        F.round(F.sum("inventory_value"), 2).alias("inventory_value")
    )
    .orderBy(F.desc("inventory_value"))
)

inventory_category.show(truncate=False)

+----------------+--------------+-----------------+-----------------+--------------------+---------------+
|category        |total_products|total_stock_units|low_stock_records|out_of_stock_records|inventory_value|
+----------------+--------------+-----------------+-----------------+--------------------+---------------+
|Ethnic Wear     |1432          |1404379          |729              |14                  |7.65090259527E9|
|Footwear        |1436          |1420797          |694              |15                  |5.4278915761E9 |
|Women's Clothing|1437          |1426999          |679              |11                  |4.56540371329E9|
|Sportswear      |1444          |1451686          |684              |7                   |3.9776192931E9 |
|Accessories     |1435          |1427626          |723              |10                  |3.72524255644E9|
|Men's Clothing  |1368          |1361377          |676              |13                  |3.68943109055E9|
|Kids' Clothing  |1448          |1469

In [0]:
replenishment_priority = (
    inventory_silver
    .filter(F.col("inventory_status").isin("Low Stock", "Out Of Stock"))
    .join(
        products_silver.select(
            "product_id",
            "category",
            "subcategory",
            "brand",
            F.col("price").alias("product_price")
        ),
        on="product_id",
        how="inner"
    )
    .select(
        "product_id",
        "category",
        "subcategory",
        "brand",
        "warehouse_id",
        "stock_quantity",
        "reorder_level",
        "inventory_status",
        "product_price"
    )
    .orderBy(
        F.when(F.col("inventory_status") == "Out Of Stock", 1).otherwise(2),
        F.col("stock_quantity").asc()
    )
)

print("Products requiring replenishment:", replenishment_priority.count())

replenishment_priority.show(20, truncate=False)

Products requiring replenishment: 4946
+----------+----------------+---------------+---------------+------------+--------------+-------------+----------------+-------------+
|product_id|category        |subcategory    |brand          |warehouse_id|stock_quantity|reorder_level|inventory_status|product_price|
+----------+----------------+---------------+---------------+------------+--------------+-------------+----------------+-------------+
|1254      |Sportswear      |Track Pants    |Vivid Vogue    |WH004       |0             |64           |Out Of Stock    |3765.63      |
|494       |Men's Clothing  |Jeans          |The Cotton Edit|WH003       |0             |29           |Out Of Stock    |428.13       |
|654       |Accessories     |Belts          |Vivid Vogue    |WH003       |0             |59           |Out Of Stock    |1325.4       |
|92        |Footwear        |Flats          |Urban Loom     |WH003       |0             |30           |Out Of Stock    |2763.76      |
|2295      |Wome

In [0]:
replenishment_summary = (
    replenishment_priority
    .withColumn(
        "recommended_reorder_quantity",
        F.col("reorder_level") - F.col("stock_quantity")
    )
    .groupBy("warehouse_id", "inventory_status")
    .agg(
        F.count("*").alias("products_to_replenish"),
        F.sum("recommended_reorder_quantity").alias("recommended_units")
    )
    .orderBy("warehouse_id", "inventory_status")
)

replenishment_summary.show(truncate=False)

+------------+----------------+---------------------+-----------------+
|warehouse_id|inventory_status|products_to_replenish|recommended_units|
+------------+----------------+---------------------+-----------------+
|WH001       |Low Stock       |1282                 |44428            |
|WH001       |Out Of Stock    |19                   |1295             |
|WH002       |Low Stock       |1210                 |41325            |
|WH002       |Out Of Stock    |26                   |1552             |
|WH003       |Low Stock       |1197                 |40360            |
|WH003       |Out Of Stock    |20                   |1181             |
|WH004       |Low Stock       |1179                 |39412            |
|WH004       |Out Of Stock    |13                   |927              |
+------------+----------------+---------------------+-----------------+



In [0]:
inventory_analytics = (
    inventory_silver
    .groupBy("warehouse_id")
    .agg(
        F.count("*").alias("total_products"),
        F.sum("stock_quantity").alias("total_stock_units"),
        F.sum(
            F.when(F.col("inventory_status") == "In Stock", 1).otherwise(0)
        ).alias("in_stock_products"),
        F.sum(
            F.when(F.col("inventory_status") == "Low Stock", 1).otherwise(0)
        ).alias("low_stock_products"),
        F.sum(
            F.when(F.col("inventory_status") == "Out Of Stock", 1).otherwise(0)
        ).alias("out_of_stock_products"),
        F.round(F.sum("inventory_value"), 2).alias("inventory_value")
    )
    .join(
        replenishment_summary
        .groupBy("warehouse_id")
        .agg(
            F.sum("products_to_replenish").alias("products_requiring_replenishment"),
            F.sum("recommended_units").alias("recommended_reorder_units")
        ),
        on="warehouse_id",
        how="left"
    )
    .orderBy("warehouse_id")
)

inventory_analytics.show(truncate=False)

+------------+--------------+-----------------+-----------------+------------------+---------------------+---------------+--------------------------------+-------------------------+
|warehouse_id|total_products|total_stock_units|in_stock_products|low_stock_products|out_of_stock_products|inventory_value|products_requiring_replenishment|recommended_reorder_units|
+------------+--------------+-----------------+-----------------+------------------+---------------------+---------------+--------------------------------+-------------------------+
|WH001       |10000         |2478646          |8699             |1282              |19                   |7.79406648579E9|1301                            |45723                    |
|WH002       |10000         |2489808          |8764             |1210              |26                   |7.72768105369E9|1236                            |42877                    |
|WH003       |10000         |2488973          |8783             |1197              |20    

In [0]:
inventory_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.inventory_analytics")
    

In [0]:
replenishment_priority.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.inventory_replenishment")

In [0]:
for table_name in [
    "inventory_analytics",
    "inventory_replenishment"
]:
    df = spark.table(f"workspace.fashion_gold.{table_name}")
    print(f"{table_name}: {df.count()} records")

inventory_analytics: 4 records
inventory_replenishment: 4946 records
